<a href="https://colab.research.google.com/github/sumpumm/Feature-Engineering/blob/main/Column%20Transformer%20in%20sklearn/columnTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Machine Learning from scratch/Feature Engineering-04/Column Transformer in sklearn/covid_toy.csv')

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [7]:
df.isna().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

In [9]:
x_train, x_test, y_train, y_test = train_test_split(df.drop('has_covid',axis=1),df['has_covid'],test_size=0.2,random_state=0)

In [10]:
x_train

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


In [35]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


#without using column transformer

In [15]:
si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])
x_test_fever = si.fit_transform(x_test[['fever']])

x_train_fever.shape

(80, 1)

In [17]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

In [18]:
x_test_cough = oe.fit_transform(x_test[['cough']])

In [26]:
ohe= OneHotEncoder(drop='first',sparse_output=False) #sparse_output=False will give np array
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

x_test_gender_city = ohe.fit_transform(x_test[['gender','city']])

x_train_gender_city.shape

(80, 4)

In [27]:
# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape

(80, 1)

In [29]:
x_train_transformed = np.concatenate((x_train_age, x_train_fever, x_train_gender_city, x_train_cough), axis=1)
x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

x_train_transformed.shape

(80, 7)

In [33]:
df_transformed = pd.DataFrame(x_train_transformed)

In [34]:
df_transformed.head()

,0,1,2,3,4,5,6
0,22.0,99.0,0.0,0.0,0.0,0.0,0.0
1,56.0,104.0,0.0,0.0,0.0,0.0,1.0
2,31.0,98.0,0.0,0.0,1.0,0.0,0.0
3,75.0,104.0,0.0,1.0,0.0,0.0,1.0
4,72.0,99.0,1.0,0.0,0.0,0.0,0.0


#Using Column transformer

In [36]:
from sklearn.compose import ColumnTransformer

In [37]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
], remainder='passthrough') #passthrough/drop -> this means the columns in which no transformations are applied and either left as
                                                                          # it is or dropped


In [38]:
x_train_new= transformer.fit_transform(x_train)

In [40]:
df_new = pd.DataFrame(x_train_new)

In [42]:
df_new.columns = transformer.get_feature_names_out()
df_new.head()

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,99.0,0.0,0.0,0.0,0.0,0.0,22.0
1,104.0,1.0,0.0,0.0,0.0,0.0,56.0
2,98.0,0.0,0.0,0.0,1.0,0.0,31.0
3,104.0,1.0,0.0,1.0,0.0,0.0,75.0
4,99.0,0.0,1.0,0.0,0.0,0.0,72.0
